In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_CRRI_Mathura_Road_Delhi_IMD_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,306.0,130.0,174.0,NaN,210.0,237.0,NaN,28.0,NaN,NaN,337.0,255.0
1,2,326.0,197.0,81.0,91.0,159.0,NaN,120.0,NaN,NaN,NaN,337.0,256.0
2,3,322.0,143.0,NaN,NaN,256.0,102.0,86.0,NaN,NaN,NaN,374.0,226.0
3,4,344.0,213.0,89.0,153.0,297.0,199.0,37.0,44.0,NaN,NaN,358.0,140.0
4,5,303.0,116.0,86.0,223.0,257.0,228.0,50.0,42.0,NaN,NaN,353.0,135.0
5,6,NaN,113.0,NaN,152.0,253.0,154.0,55.0,52.0,NaN,NaN,329.0,154.0
6,7,306.0,149.0,153.0,173.0,NaN,NaN,34.0,NaN,NaN,NaN,359.0,208.0
7,8,325.0,109.0,115.0,231.0,NaN,NaN,35.0,NaN,NaN,NaN,NaN,285.0
8,9,347.0,101.0,NaN,NaN,152.0,158.0,74.0,38.0,NaN,NaN,328.0,123.0
9,10,219.0,300.0,136.0,NaN,144.0,146.0,112.0,NaN,NaN,NaN,317.0,230.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,306.000000,130.0000,174.000000,145.6,210.000000,237.0000,67.692308,27.888889,15.0,91.25,337.0,255.0
1,2,326.000000,197.0000,81.000000,145.6,159.000000,127.3125,67.692308,27.888889,15.0,91.25,337.0,256.0
2,3,322.000000,143.0000,120.451613,145.6,256.000000,102.0000,86.000000,27.888889,15.0,91.25,374.0,226.0
3,4,344.000000,213.0000,89.000000,153.0,297.000000,199.0000,67.692308,27.888889,15.0,91.25,358.0,140.0
4,5,303.000000,116.0000,86.000000,145.6,257.000000,228.0000,50.000000,27.888889,15.0,91.25,353.0,135.0
5,6,283.757576,113.0000,120.451613,152.0,253.000000,154.0000,55.000000,27.888889,15.0,91.25,329.0,154.0
6,7,306.000000,149.0000,153.000000,173.0,184.636364,127.3125,67.692308,27.888889,15.0,91.25,359.0,208.0
7,8,325.000000,109.0000,115.000000,145.6,184.636364,127.3125,67.692308,27.888889,15.0,91.25,288.6,285.0
8,9,347.000000,101.0000,120.451613,145.6,152.000000,158.0000,74.000000,27.888889,15.0,91.25,328.0,123.0
9,10,219.000000,300.0000,136.000000,145.6,144.000000,146.0000,67.692308,27.888889,15.0,91.25,317.0,230.0
